# 05 · Build the report

Assembles `USAGE_REPORT.md` from the metrics the other notebooks wrote and the figures they saved.
Nothing is recomputed here — if a number is wrong, it is wrong in the notebook that produced it,
which is the point of the hand-off.

The document's structure is a contract, because the deck is generated from it:

```
# Part — <part title>
## S07 · <slide title>
**Takeaway:** the line that goes on the slide
![...](figures/<name>.png)
*Figure: caption, always with its n.*
| the figure's own data table |
**Speaker notes:** spoken, not shown
**Confidence:** measured | indicative | speculative
```

Run notebooks 00 → 04 first; this one fails loudly if a section is missing.

In [1]:
# Put the analysis package on the path no matter where Jupyter was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "analysis" / "bloombot_analysis").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "analysis"))

import pandas as pd

from bloombot_analysis import charts, load, metrics, privacy, report, sessions, topics
from bloombot_analysis.config import CONFIG, SURFACE_LABELS, TOPICS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("as of:", CONFIG.as_of)
print("legacy db :", CONFIG.legacy_db, "(exists)" if CONFIG.legacy_db.exists() else "(missing)")
print("current db:", CONFIG.current_db, "(exists)" if CONFIG.current_db.exists() else "(missing)")
print("output    :", CONFIG.out_dir)

as of: 2026-09-25
legacy db : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/legacy.db (exists)
current db: /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/current.db (exists)
output    : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out


In [2]:
everything = metrics.load()
metrics.require(everything, "dataset", "rows", "sessions", "elapsed_days")

dataset = everything["dataset"]
volume = everything.get("volume", {})
shape = everything.get("shape", {})
topic_metrics = everything.get("topics", {})
cost = everything.get("cost", {})

fmt_int, fmt_num, fmt_share = report.fmt_int, report.fmt_num, report.fmt_share
fmt_count_of, md_table = report.fmt_count_of, report.markdown_table

elapsed = dataset["elapsed_days"]
total_days = dataset["total_days"]
current_label = dataset["current_term"]
comparison_label = dataset["comparison_term"]
as_of = dataset["as_of"]

doc = report.Report(
    title=f"Bloombot usage, {comparison_label} through {current_label}",
    subtitle=(
        f"An AI course assistant across four courses · data through {as_of} "
        f"(day {elapsed} of {total_days} of {current_label})"
    ),
    preamble=(
        "This document is the source for the talk: one `## S..` heading per slide, a takeaway line "
        "that goes on the slide, a figure with its n, the figure's own data table, speaker notes, "
        "and a **Confidence** field — `measured` for a count out of the data, `indicative` for a "
        "real pattern on an n small enough to move, `speculative` for our own reading of it.\n\n"
        f"Generated from `analysis/notebooks/` on the dataset described in Part — Method. "
        f"{current_label} is **in progress**: every number from it is a partial-term number."
    ),
)
print(doc.title)

Bloombot usage, Fall 2025 through Fall 2026


In [3]:
doc.part("What Bloombot is")

doc.slide(
    "An AI course assistant, in the places students already are",
    takeaway="One assistant, grounded in one course's own materials, reachable three ways.",
    body=(
        "Bloombot answers a student's questions about a specific course — its schedule, its "
        "assignments, its setup instructions, its policies — in the student's own words, at "
        "whatever hour they ask. It is not a general chatbot: it knows which course is being "
        "asked about, and it answers from that course's own material.\n\n"
        "- **In Discord**, where the course already lives: a shared channel, or the student's own "
        "private channel.\n"
        "- **On the web**, through a course chat page reached by an emailed sign-in link. New this "
        "fall.\n"
        "- **Through a chat assistant** (Claude, or any MCP-capable client) connected to the "
        "course, so a student already working inside an AI assistant can ask the course bot "
        "without leaving it. Also new this fall.\n\n"
        "All three reach the same assistant, with the same course instructions and the same memory "
        "of what that student has asked before."
    ),
    notes=(
        "Show a screenshot of each interface here. The point to land: the student does not go "
        "anywhere special to use it."
    ),
)

doc.slide(
    "How it answers",
    takeaway="A commercially hosted model, per-student memory, everything logged, spending capped.",
    body=(
        "The model is commercially hosted — currently OpenAI's `gpt-4.1`, configurable per course. "
        "Nothing is trained or fine-tuned here: the intelligence is rented, and the "
        "course-specific part is the instructions and materials the instructor supplies. A cheaper "
        "model (`gpt-4o-mini`) does the offline topic classification behind this report, never the "
        "student-facing answers.\n\n"
        "- **Conversation memory is per student, per course, per interface.** A follow-up arrives "
        "with that student's whole prior history in the course already in context: the hosted "
        "conversation is carried by id rather than re-sent as a handful of recent turns, so it is "
        "not truncated by us, and it survives restarts and days between visits. The original "
        "Python bot held this in memory only — a restart wiped it. That changed this fall.\n"
        "- **Every message, both directions, is logged** to the instructor's own database. That is "
        "what makes this report possible, and what an instructor relies on when reading a "
        "transcript.\n"
        "- **A daily per-student request cap and a per-course spending cap** mean a runaway loop "
        "or an enthusiastic student cannot run up an unbounded bill."
    ),
    notes=(
        "If asked about hallucination: the assistant is grounded in the course's own materials and "
        "instructions, and the transcript is readable by the instructor — which is the actual "
        "check, rather than a claim about the model."
    ),
)

doc.slide(
    "How it knows who is asking, and which course they are in",
    takeaway="Discord category, Discord role, or roster import — and we deliberately do not use roster import.",
    body=(
        "- **By Discord category.** A message in a course's category is a question about that "
        "course. The original mechanism, and still the most used.\n"
        "- **By Discord role.** A role on the server marks a member as enrolled, which is what "
        "grants access to that course's assistant.\n"
        "- **By roster import.** A CSV of enrolled students can create enrolments directly. "
        "**We have deliberately not used roster import for our own university courses.** A roster "
        "is an education record, and handing one to a third-party-hosted system raises FERPA "
        "questions we chose not to answer by acting first. Students arrive by joining the Discord "
        "server or by following a join link instead, which keeps the institutional roster out of "
        "the system entirely. The capability exists for instructors whose institutions have "
        "cleared it."
    ),
    notes=(
        "Say the FERPA point plainly and early — it is the question the audience is already "
        "forming, and answering it before it is asked is worth more than any chart in the deck."
    ),
)

doc.slide(
    "Other instructors can run their own courses on it",
    takeaway="Multi-tenant, sign-in by emailed link, and account creation is gated on approval.",
    body=(
        "Another instructor can create an account, set up their own organization and courses, "
        "connect their own Discord server, and run their own assistant, with their data kept "
        "separate from everyone else's. Sign-in is by emailed link; there is no password to "
        "manage.\n\n"
        "Account creation is **gated on my approval**, deliberately. Every tenant's usage spends "
        "real money against a hosted model, and every new tenant brings student data into the "
        "system — so admission is a decision, not a signup form. Approvals are recorded."
    ),
    notes="This is the slide where people ask 'can I have one'. The answer is yes, after a conversation.",
)

doc.slide(
    "How it was built",
    takeaway="One developer, no funding: hand-built in Python, then rebuilt and extended with Claude Code for Fall 2026.",
    body=(
        "- **One developer, no funding.** No grant, no team, no institutional project.\n"
        "- **Built by hand first, in Python**: a Discord bot with a SQLite message log and an "
        "analytics notebook, run across four courses over the past academic year. That version "
        "produced most of the historical data in this report.\n"
        "- **Rebuilt and substantially extended for Fall 2026 with Claude Code**, ported to a "
        "TypeScript/JavaScript stack: the web interface, the chat-assistant (MCP) interface, "
        "accounts and sign-in, multi-tenancy, per-course instructions and attachments, cost "
        "tracking, data retention and deletion, and an administration CLI. The Discord bot's "
        "behaviour was preserved and its history imported, so a year of prior conversations and "
        "this fall's sit in one database — which is what makes the comparisons later in this deck "
        "possible at all."
    ),
    notes=(
        "Worth one sentence on what this says about the cost of building this kind of tool now — "
        "it is the most transferable thing in the talk."
    ),
)
print(doc.slide_count(), "slides so far")

5 slides so far


In [4]:
doc.part("Method")

doc.slide(
    "What counts as a session",
    takeaway=(
        f"One student, one course, one interface, split after "
        f"{dataset['gap_minutes']} minutes of silence."
    ),
    body=(
        f"- **Session**: consecutive messages between one student and the bot in one course on one "
        f"interface, split whenever more than **{dataset['gap_minutes']} minutes** of silence "
        f"passes. The stored conversation record is deliberately not the unit — on the web a "
        f"conversation can hold a whole term.\n"
        "- **Prompt**: one student message. Bot replies are not counted, or every session would "
        "look twice as deep as it is.\n"
        "- **Active student**: at least one prompt in the period.\n"
        f"- **Excluded**: instructor and test accounts "
        f"({fmt_int(dataset.get('excluded_account_rows', 0))} messages), and anything a student "
        f"has asked to have deleted.\n"
        f"- All times are local ({dataset.get('timezone', 'local')}), so 'by hour of day' means "
        f"the hour the student was awake."
    ),
    table=md_table(pd.DataFrame(shape.get("sensitivity", []))),
    notes=(
        "The table is the sensitivity check: the same headline numbers re-derived at 15, 30 and 60 "
        "minutes. If they barely move, say so in one line and move on — that is the point of "
        "showing it."
    ),
)

doc.slide(
    "Where the data comes from",
    takeaway=(
        f"{fmt_int(dataset['rows'])} messages from two databases, reconciled into one, "
        f"{fmt_int(dataset['duplicates_dropped'])} duplicates dropped."
    ),
    body=(
        "Bloombot's history spans two data models: the Python bot's log up to Fall 2026, and the "
        "current platform's. Some of the old log was imported into the new database, so the same "
        "message can exist in both — those are matched on content and timestamp and counted once. "
        "Students are joined across the two by their Discord identity, so someone who used the bot "
        "last year and again this fall is one person, not two."
    ),
    table=md_table(
        pd.DataFrame(
            [
                ("Messages in the pre-Fall-2026 database", fmt_int(dataset["legacy_rows"])),
                ("Messages in the current platform database", fmt_int(dataset["current_rows"])),
                ("Dropped as already-imported duplicates", fmt_int(dataset["duplicates_dropped"])),
                ("Dropped as instructor/test accounts", fmt_int(dataset["excluded_account_rows"])),
                ("Messages analysed", fmt_int(dataset["rows"])),
                ("Sessions", fmt_int(dataset["sessions"])),
                ("Distinct students", fmt_int(dataset["students"])),
                ("First message", str(dataset["first_message"])[:10]),
                ("Last message", str(dataset["last_message"])[:10]),
            ],
            columns=["", "Count"],
        )
    ),
    notes="If anyone asks whether messages are double-counted: this row-by-row table is the answer.",
)

doc.slide(
    f"{current_label} is not finished",
    takeaway=(
        f"This term is {fmt_share(dataset['fraction_elapsed'])} elapsed — day {elapsed} of "
        f"{total_days}. Every {current_label} number here is partial."
    ),
    body=(
        f"{current_label} runs {dataset['current_start']} → {dataset['current_end']}. The data "
        f"stops at {as_of}. Two consequences the rest of the deck is built around:\n\n"
        f"1. **Nothing from this term is compared against a complete prior term.** Every "
        f"year-over-year comparison cuts both terms to the same first **{elapsed} days** of their "
        f"own term.\n"
        "2. **The new interfaces launched this term.** Web and chat-assistant volume measures our "
        "own release as much as it measures student behaviour, and every chart that mixes "
        "interfaces says so."
    ),
    notes=(
        "This slide is the one that earns the right to show the rest. Do not skip it, and do not "
        "apologise for it — a partial term honestly labelled is worth more than a full term "
        "quietly implied."
    ),
)
print(doc.slide_count(), "slides so far")

8 slides so far


In [5]:
doc.part("Findings")
figures = CONFIG.out_dir / "figures"

adoption_rows = pd.DataFrame(volume.get("adoption", []))
if not adoption_rows.empty:
    adoption_rows = pd.DataFrame(
        {
            "Course": adoption_rows["course"],
            "Enrolled": adoption_rows["enrolled"],
            "Used the bot": adoption_rows["active"],
            "Share": [
                f"{int(a)} / {int(e)}" if pd.notna(e) else "—"
                for a, e in zip(adoption_rows["active"], adoption_rows["enrolled"])
            ],
        }
    )

doc.slide(
    "Adoption: who used it at all",
    takeaway=(
        f"{fmt_count_of(volume.get('adoption_total_active', 0), volume.get('adoption_total_enrolled', 0))} "
        f"enrolled students used the bot at least once this term so far."
    ),
    figure=figures / volume.get("figures", {}).get("adoption", "adoption_by_course.png"),
    figure_caption=(
        f"Distinct students with at least one prompt, {current_label} through day {elapsed} "
        f"(n = {fmt_int(volume.get('adoption_total_active', 0))} students)."
    ),
    table=md_table(adoption_rows),
    body=(
        "Adoption has a denominator that means something, which a raw message count does not. "
        "Enrolment exists only from Fall 2026 — the old bot had no roster — so this cannot be "
        "computed for prior terms at all."
    ),
    notes="Expect the question 'is that good?'. There is no benchmark; say so, and give the range across courses.",
    confidence="measured",
)

doc.slide(
    "Volume over the whole period",
    takeaway=(
        f"Sessions peak in the first weeks of a term and around deadlines; the busiest week in the "
        f"data had {fmt_int(volume.get('weekly_peak_sessions', 0))} sessions."
    ),
    figure=figures / volume.get("figures", {}).get("weekly", "weekly_sessions_by_course.png"),
    figure_caption=(
        f"Sessions per week by course, {str(dataset['first_message'])[:10]} → "
        f"{str(dataset['last_message'])[:10]} (n = {fmt_int(dataset['sessions'])} sessions). "
        "The dashed rule marks the start of Fall 2026, when the web and chat-assistant interfaces "
        "launched."
    ),
    body=(
        "Points, not a trend line: with this many weeks a fitted line would assert more than the "
        "data supports. The shape — a start-of-term spike, then deadline-shaped bumps — is the "
        "most legible thing in the dataset."
    ),
    notes="Walk the audience along the line and name the weeks; the shape does the argument for you.",
    confidence="measured",
)

surface_rows = pd.DataFrame(volume.get("surface_split", []))
if not surface_rows.empty:
    surface_rows = surface_rows[["surface_label", "sessions", "prompts", "students"]].rename(
        columns={
            "surface_label": "Interface",
            "sessions": "Sessions",
            "prompts": "Prompts",
            "students": "Students",
        }
    )

doc.slide(
    "Where students talked to it",
    takeaway="Discord still carries most of it; the two new interfaces are being used, at small numbers.",
    figure=figures / volume.get("figures", {}).get("surfaces", "sessions_by_surface.png"),
    figure_caption=(
        f"Sessions by interface, {current_label} through day {elapsed} "
        f"(n = {fmt_int(surface_rows['Sessions'].sum()) if not surface_rows.empty else 0} sessions)."
    ),
    table=md_table(surface_rows),
    body=(
        "Web and the chat assistant have existed for three weeks. Their share is a fact about our "
        "launch, not a preference students expressed over a year."
    ),
    notes="Resist reading a 'preference' into this. The honest claim is that both new doors got used at all.",
    confidence="indicative",
)

comparison_frame = pd.DataFrame(volume.get("comparison", {}))
doc.slide(
    f"{comparison_label} against {current_label}, like for like",
    takeaway=(
        f"Both terms cut to their first {elapsed} days, so this compares behaviour rather than "
        "the calendar."
    ),
    figure=figures / volume.get("figures", {}).get("comparison", "term_comparison.png"),
    figure_caption=(
        f"Sessions, distinct students and prompts in the first {elapsed} days of each term."
    ),
    table=md_table(comparison_frame, index_label="Measure"),
    body=(
        "The comparison that is *not* like-for-like is the interface split: Discord is the only "
        "row that existed in both terms, and the next slide separates it out for that reason."
    ),
    notes=(
        "If this shows growth, the honest phrasing is 'more sessions in the same span of term', not "
        "'usage is up X%' — three weeks is three weeks."
    ),
    confidence="indicative",
)

doc.slide(
    "The same window, by interface",
    takeaway="Discord is the only interface with both terms behind it; the others start at zero by construction.",
    figure=figures / volume.get("figures", {}).get("comparison_by_surface", "term_comparison_by_surface.png"),
    figure_caption=f"Sessions by interface in the first {elapsed} days of each term.",
    table=md_table(pd.DataFrame(volume.get("comparison_by_surface", {})), index_label="Interface"),
    body=(
        "Read the Discord pair as the behavioural comparison and the other two as a launch record. "
        "A combined total would blur exactly that distinction, which is why it is not shown here."
    ),
    notes="This is the slide that prevents someone quoting a growth number that is really a feature release.",
    confidence="measured",
)

doc.slide(
    "What a typical session looks like",
    takeaway=(
        f"Median {fmt_num(shape.get('median_prompts'), 0)} prompts per session "
        f"(IQR {fmt_num((shape.get('iqr_prompts') or [None, None])[0], 0)}–"
        f"{fmt_num((shape.get('iqr_prompts') or [None, None])[1], 0)}); "
        f"{fmt_count_of(shape.get('single_prompt_sessions'), shape.get('sessions'))} sessions are a "
        "single question."
    ),
    figure=figures / shape.get("figures", {}).get("prompts", "prompts_per_session.png"),
    figure_caption=(
        f"Distribution of student messages per session (n = {fmt_int(shape.get('sessions'))} "
        "sessions). The median is marked."
    ),
    body=(
        f"Mean {fmt_num(shape.get('mean_prompts'))}, median {fmt_num(shape.get('median_prompts'), 0)} — "
        "the gap is the long tail of a few deep sessions, which is why the median leads. "
        f"{fmt_int(shape.get('long_sessions'))} sessions ran to five prompts or more."
    ),
    notes=(
        "The one-question majority is the real finding here, and it argues the bot is used as a "
        "reference, not as a tutor. Say that as a reading, not as a measurement."
    ),
    confidence="measured",
)

doc.slide(
    "How long a session lasts, and when it happens",
    takeaway=(
        f"Median {fmt_num(shape.get('median_duration'))} minutes; "
        f"{fmt_count_of(shape.get('after_hours_sessions'), shape.get('total_sessions'))} sessions "
        "start between 18:00 and 08:00."
    ),
    figure=figures / shape.get("figures", {}).get("hours", "sessions_by_hour.png"),
    figure_caption=(
        f"Sessions by the hour they started, local time "
        f"(n = {fmt_int(shape.get('total_sessions'))} sessions)."
    ),
    body=(
        "The after-hours share is the clearest argument for an always-on assistant in the whole "
        "dataset: these are questions that would otherwise have waited for the next class or gone "
        "unasked."
    ),
    notes=(
        "Careful: 'would otherwise have gone unasked' is an inference. The measured part is the "
        "hour distribution."
    ),
    confidence="measured",
)

returns = shape.get("returns", {})
doc.slide(
    "Do they come back?",
    takeaway=(
        f"{fmt_count_of(returns.get('returned'), returns.get('students'))} students who used the "
        f"bot used it again on another day."
    ),
    body=(
        f"Median active days per student: {fmt_num(returns.get('median_active_days'))}. Reported as "
        "counts rather than a retention curve on purpose — a curve on a cohort this size implies a "
        "precision the data does not have."
    ),
    notes="A returning student is the closest thing to a satisfaction signal we have. It is still not one.",
    confidence="measured",
)

topic_rows = pd.DataFrame(topic_metrics.get("counts", []))
if not topic_rows.empty:
    topic_rows = topic_rows.rename(columns={"topic": "Topic", "sessions": "Sessions"})
agreement = topic_metrics.get("agreement", {}) or {}
agreement_line = (
    f"Hand-audit: classifier and human agreed on {fmt_count_of(agreement.get('agreed'), agreement.get('checked'))} "
    "sampled sessions."
    if agreement.get("checked")
    else "**No hand-audit has been recorded yet** — fill in `topic_audit_completed.csv` and re-run "
    "notebook 03 before presenting these charts."
)

doc.slide(
    "What students asked about",
    takeaway=(
        f"The largest category is {topic_metrics.get('top_topic', '—')} "
        f"({fmt_int(topic_metrics.get('top_topic_sessions'))} sessions); "
        f"{fmt_share(topic_metrics.get('other_share'))} of sessions fall outside the label set."
    ),
    figure=figures / topic_metrics.get("figures", {}).get("overall", "topics_overall.png"),
    figure_caption=(
        f"Sessions per topic, all courses (n = {fmt_int(dataset.get('sessions'))} sessions, "
        f"classified by {topic_metrics.get('method', 'keyword')})."
    ),
    table=md_table(topic_rows),
    body=(
        f"{agreement_line}\n\nA large *Other* share is itself a finding about what the nine "
        "labels miss, not a gap to hide."
    ),
    notes=(
        "Name the classifier and its agreement rate out loud. A topic chart from an unaudited "
        "classifier is an assertion."
    ),
    confidence="indicative",
)

doc.slide(
    "The mix differs by course",
    takeaway="Programming courses ask about setup and concepts; project courses ask about teams and deadlines.",
    figure=figures / topic_metrics.get("figures", {}).get("by_course", "topics_by_course.png"),
    figure_caption=(
        "Sessions by topic and course. "
        + (topic_metrics.get("suppression_note") or "No cells suppressed.")
    ),
    table=md_table(pd.DataFrame(topic_metrics.get("by_course", {})), index_label="Topic"),
    body=(
        "Cells backed by fewer than five distinct students are suppressed and drawn empty: in "
        "cohorts this small, a cell of one is effectively a named individual."
    ),
    notes="The most quotable finding in the deck. Pick the two courses with the sharpest contrast and stop there.",
    confidence="indicative",
)

doc.slide(
    "Did the new interfaces change what gets asked?",
    takeaway="Suggestive at best: three weeks of one term against three weeks of another.",
    figure=figures / topic_metrics.get("figures", {}).get("by_term", "topics_by_term.png"),
    figure_caption=f"Topics in the first {elapsed} days of each term.",
    table=md_table(pd.DataFrame(topic_metrics.get("by_term", {})), index_label="Topic"),
    body=(
        "The hypothesis worth stating: a private web page invites questions a student might not ask "
        "in a shared Discord channel. This chart is consistent with that and does not establish it."
    ),
    notes="Offer it as the thing to measure next term, not as a result.",
    confidence="speculative",
)

quotes = topic_metrics.get("quotes", [])
quote_body = "\n\n".join(
    f"**{q.get('topic', '—')}** · {q.get('course', '')}\n\n> {q.get('quote', '')}"
    for q in quotes[:4]
) or "_No quote candidates available._"
doc.slide(
    "In their words",
    takeaway="Two or three real exchanges do more for an audience than any chart here.",
    body=(
        "Candidates below are mechanically redacted (emails, mentions, links and long numbers "
        "removed) and **must be paraphrased by a human before they go on a slide**.\n\n"
        + quote_body
    ),
    notes="Read one aloud. Do not put a student's exact words on a screen without paraphrasing them.",
    confidence="measured",
)

Slide(number=20, title='In their words', takeaway='Two or three real exchanges do more for an audience than any chart here.', body="Candidates below are mechanically redacted (emails, mentions, links and long numbers removed) and **must be paraphrased by a human before they go on a slide**.\n\n**Professor & office hours** · Web Design\n\n> Student: when are office hours this week Bot: That error usually means a missing dependency. Try these three steps in order.\n\n**Technical setup & tools** · Agile Software Development & DevOps\n\n> Student: docker won't start on port 3000, is that normal Bot: Here's what the course materials say about that, and where to look next.\n\n**Resources & references** · Web Design\n\n> Student: is there a tutorial you recommend for this Bot: The syllabus covers this: the relevant deadline and policy are below.\n\n**Grades & assessment** · Web Design\n\n> Student: what is the rubric for the midterm Bot: Here's what the course materials say about that, and wh

In [6]:
if cost.get("rows"):
    per_session = cost.get("usd_per_session")
    doc.slide(
        "What it costs to run",
        takeaway=(
            f"${fmt_num(cost.get('total_usd'), 2)} of model spend over the ledger window — about "
            f"${fmt_num(per_session, 3)} per session."
        ),
        figure=figures / cost.get("figure", "cost_by_course.png"),
        figure_caption=(
            f"Model spend by course, {str(cost.get('window_start'))[:10]} → "
            f"{str(cost.get('window_end'))[:10]} (n = {fmt_int(cost.get('rows'))} metered calls)."
        ),
        table=md_table(
            pd.DataFrame(cost.get("by_course", [])).rename(
                columns={
                    "course": "Course",
                    "usd": "Spend (USD)",
                    "calls": "Metered calls",
                    "input_tokens": "Input tokens",
                    "output_tokens": "Output tokens",
                }
            )[["Course", "Spend (USD)", "Metered calls", "Input tokens", "Output tokens"]]
            if cost.get("by_course")
            else pd.DataFrame()
        ),
        body=(
            "The cost ledger exists only on the current platform and only for live traffic, so "
            "this is a Fall 2026 figure over the window shown — not a yearly cost, and not "
            "extrapolated to one. Per student in the window: "
            f"${fmt_num(cost.get('usd_per_student'), 2)} across "
            f"{fmt_int(cost.get('students_in_window'))} students."
        ),
        notes="If asked about a per-student-per-term cost, the answer is that the term is not over.",
        confidence="measured",
    )

doc.part("What this suggests, and what it does not")

doc.slide(
    "A reading of the numbers",
    takeaway="Used as an always-available reference at the edges of the day, not as a tutor.",
    body=(
        "Stated as hypotheses, with the evidence attached:\n\n"
        f"1. **Reference, not tutoring.** A session is short — median "
        f"{fmt_num(shape.get('median_prompts'), 0)} prompts, "
        f"{fmt_count_of(shape.get('single_prompt_sessions'), shape.get('sessions'))} of them a "
        f"single question — and the largest topic is "
        f"{topic_metrics.get('top_topic', 'unclear')}. That pattern fits a look-it-up habit more "
        "than a study-with-me one.\n"
        f"2. **It fills the hours nobody staffs.** "
        f"{fmt_count_of(shape.get('after_hours_sessions'), shape.get('total_sessions'))} sessions "
        "start between 18:00 and 08:00.\n"
        "3. **Adoption is broad but shallow.** A substantial share of each roster tried it; a "
        "smaller group returns repeatedly. Whether the shallow group got what they needed or gave "
        "up is exactly what this data cannot say.\n"
        "4. **A private interface may invite different questions.** Consistent with the topic "
        "split by interface; not established by it."
    ),
    notes=(
        "This slide was written first, before the deck was built around it. If it cannot be "
        "argued from the charts, the analysis is not finished."
    ),
    confidence="speculative",
)

doc.slide(
    "What this data cannot tell you",
    takeaway="No outcomes, no signal from non-users, and a term that is three weeks old.",
    body=(
        "- **No outcome data.** Nothing links a conversation to a grade, a submission, or whether "
        "the answer was right or helpful. Every value claim in this deck is inference.\n"
        "- **No signal from non-use.** A student who never messaged the bot is invisible except as "
        "a roster row. We do not know whether they did not need it, did not know about it, or did "
        "not trust it.\n"
        f"- **A partial term.** {current_label} is {fmt_share(dataset['fraction_elapsed'])} "
        "elapsed. Deadline-driven traffic is front-loaded in this window and the heaviest weeks of "
        "the term have not happened yet.\n"
        "- **Small n throughout.** Every chart carries its own n for this reason.\n"
        "- **An unaudited-by-default classifier.** Topic labels are a model's judgement; the "
        "agreement rate is stated wherever it exists."
    ),
    notes="Saying this yourself is worth more than having it asked from the floor.",
    confidence="measured",
)

doc.slide(
    "What would make the next version of this talk stronger",
    takeaway="Three cheap instruments: a reply rating, a one-question exit survey, and deadline dates.",
    body=(
        "1. **A thumbs-up/down on each reply.** The single missing signal that would turn every "
        "inference in this deck into a measurement.\n"
        "2. **A one-question exit survey** at the end of the term, including the students who "
        "never used it — the only way to see non-use.\n"
        "3. **Assignment deadlines as data.** With due dates in the database, 'deadline-driven "
        "traffic' becomes a number instead of a shape on a chart.\n"
        "4. **Re-run this report at the end of term**, when the comparison is term-to-term and "
        "complete on both sides."
    ),
    notes="End here. The ask, if there is one, is for the survey.",
    confidence="measured",
)

path = doc.write(CONFIG.report_path)
print(f"{doc.slide_count()} slides → {path}")
print(path.read_text()[:1200])

24 slides → /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out/USAGE_REPORT.md
# Bloombot usage, Fall 2025 through Fall 2026

_An AI course assistant across four courses · data through 2026-09-25 (day 23 of 104 of Fall 2026)_

This document is the source for the talk: one `## S..` heading per slide, a takeaway line that goes on the slide, a figure with its n, the figure's own data table, speaker notes, and a **Confidence** field — `measured` for a count out of the data, `indicative` for a real pattern on an n small enough to move, `speculative` for our own reading of it.

Generated from `analysis/notebooks/` on the dataset described in Part — Method. Fall 2026 is **in progress**: every number from it is a partial-term number.

# Part — What Bloombot is

## S01 · An AI course assistant, in the places students already are

**Takeaway:** One assistant, grounded in one course's own materials, reachable three ways.

Bloombot answers a student's questions about a specific